<a href="https://colab.research.google.com/github/solive-11/dissertation-weed-detection/blob/main/01_environment_and_dataset_investigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
#from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation_weed_detection")
print(PROJECT_ROOT)

/content/drive/MyDrive/dissertation_weed_detection


In [6]:
directories = ["00_environment", "01_data/extracted", "01_data/processed", "01_data/splits", "02_notebooks", "03_configs",
               "04_checkpoints", "05_results/raw", "05_results/metrics", "05_results/tables", "06_figures", "07_logs",]

for directory in directories:
    (PROJECT_ROOT / directory).mkdir(
        parents=True,
        exist_ok=True )
print("Project structure created.")

Project structure created.


In [12]:
# experiment environment
import platform
import sys
import torch
from pathlib import Path
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,}

if torch.cuda.is_available():
    environment["gpu"] = torch.cuda.get_device_name(0)

environment

{'python': '3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35',
 'pytorch': '2.11.0+cpu',
 'cuda_available': False,
 'cuda_version': None}

In [13]:
#reproducibility
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [14]:
!pip install -q \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    scikit-learn \
    opencv-python-headless \
    pillow \
    pyyaml \
    tqdm

#Dataset: MH-Weed16

In [15]:
RAW_DIR = Path("/content/drive/MyDrive/dissertation_weed_detection/01_data/raw/MH-Weed16")

print("Dataset exists:", RAW_DIR.exists())
print("\nContents:\n")

for path in sorted(RAW_DIR.rglob("*")):
    relative = path.relative_to(RAW_DIR)

    if path.is_file():
        print(f"FILE  : {relative}")
    elif path.is_dir():
        print(f"DIR   : {relative}")

Dataset exists: True

Contents:

DIR   : Crop with Weeds
FILE  : Crop with Weeds/Canon Camera_Clicks.zip
FILE  : Crop with Weeds/iPhone_Clicks.zip
FILE  : Crop with Weeds/intel Real Sense Depth_Annotations.zip
FILE  : Crop with Weeds/intel Real Sense Depth_Clicks.zip
DIR   : Individual Weed Species
FILE  : Individual Weed Species/16 Classes of Weed_Species.zip
FILE  : UAV.zip


In [17]:
import zipfile
zip_files = sorted(RAW_DIR.rglob("*.zip"))
print(f"Found {len(zip_files)} ZIP files:\n")
for zip_path in zip_files:
    print("=" * 80)
    print(f"ZIP: {zip_path.relative_to(RAW_DIR)}")

    with zipfile.ZipFile(zip_path, "r") as z:
        files = z.namelist()

        print(f"Number of entries: {len(files)}")
        print("\nFirst 20 entries:")

        for item in files[:5]:
            print("  ", item)

Found 6 ZIP files:

ZIP: Crop with Weeds/Canon Camera_Clicks.zip
Number of entries: 346

First 20 entries:
   Canon Camera_Clicks/
   Canon Camera_Clicks/IMG_0132.JPG
   Canon Camera_Clicks/IMG_0133.JPG
   Canon Camera_Clicks/IMG_0134.JPG
   Canon Camera_Clicks/IMG_0135.JPG
ZIP: Crop with Weeds/iPhone_Clicks.zip
Number of entries: 577

First 20 entries:
   iPhone_Clicks/
   iPhone_Clicks/IMG_0047.JPG
   iPhone_Clicks/IMG_0048.JPG
   iPhone_Clicks/IMG_0049.JPG
   iPhone_Clicks/IMG_0050.JPG
ZIP: Crop with Weeds/intel Real Sense Depth_Annotations.zip
Number of entries: 19972

First 20 entries:
   intel Real Sense Depth_Annotations/
   intel Real Sense Depth_Annotations/Json/
   intel Real Sense Depth_Annotations/Json/090800iy3emo472414_684.json
   intel Real Sense Depth_Annotations/Json/090805j6869f452411_137.json
   intel Real Sense Depth_Annotations/Json/090807o0g127452411_140.json
ZIP: Crop with Weeds/intel Real Sense Depth_Clicks.zip
Number of entries: 6657

First 20 entries:
   intel

In [18]:
from collections import Counter
for zip_path in zip_files:
    print("=" * 80)
    print(f"{zip_path.relative_to(RAW_DIR)}")

    with zipfile.ZipFile(zip_path, "r") as z:
        files = [
            name for name in z.namelist()
            if not name.endswith("/")]

        extensions = Counter()

        for name in files:
            suffix = Path(name).suffix.lower()
            extensions[suffix if suffix else "[no extension]"] += 1

        print(f"Files: {len(files)}")
        print("File types:")

        for ext, count in sorted(extensions.items()):
            print(f"  {ext:15} {count}")

Crop with Weeds/Canon Camera_Clicks.zip
Files: 345
File types:
  .jpg            345
Crop with Weeds/iPhone_Clicks.zip
Files: 576
File types:
  .jpg            576
Crop with Weeds/intel Real Sense Depth_Annotations.zip
Files: 19968
File types:
  .json           6656
  .txt            6656
  .xml            6656
Crop with Weeds/intel Real Sense Depth_Clicks.zip
Files: 6656
File types:
  .jpeg           6656
Individual Weed Species/16 Classes of Weed_Species.zip
Files: 19141
File types:
  .png            19141
UAV.zip
Files: 282
File types:
  .jpg            282


In [19]:
import json

annotation_zip = (
    RAW_DIR
    / "Crop with Weeds"
    / "intel Real Sense Depth_Annotations.zip")

with zipfile.ZipFile(annotation_zip, "r") as z:

    # Pick the first file of each annotation type
    json_file = next(
        name for name in z.namelist()
        if name.lower().endswith(".json"))

    xml_file = next(
        name for name in z.namelist()
        if name.lower().endswith(".xml"))

    txt_file = next(
        name for name in z.namelist()
        if name.lower().endswith(".txt"))

    print("JSON file:")
    print(json_file)

    print("\nXML file:")
    print(xml_file)

    print("\nTXT file:")
    print(txt_file)

    # Read JSON
    json_content = z.read(json_file).decode("utf-8")

    print("\n" + "=" * 80)
    print("JSON CONTENT:")
    print("=" * 80)
    print(json_content[:3000])

    # Read TXT
    txt_content = z.read(txt_file).decode("utf-8")

    print("\n" + "=" * 80)
    print("TXT CONTENT:")
    print("=" * 80)
    print(txt_content[:3000])

    # Read XML
    xml_content = z.read(xml_file).decode("utf-8")

    print("\n" + "=" * 80)
    print("XML CONTENT:")
    print("=" * 80)
    print(xml_content[:3000])

JSON file:
intel Real Sense Depth_Annotations/Json/090800iy3emo472414_684.json

XML file:
intel Real Sense Depth_Annotations/PASCAL_VOC/090800iy3emo472414_684.xml

TXT file:
intel Real Sense Depth_Annotations/YOLO_darknet/090800iy3emo472414_684.txt

JSON CONTENT:
[
    {
        "class_id": 3,
        "x_center": 0.89375,
        "y_center": 0.3787037037037037,
        "width": 0.21145833333333333,
        "height": 0.5314814814814814
    },
    {
        "class_id": 3,
        "x_center": 0.1421875,
        "y_center": 0.39537037037037037,
        "width": 0.27708333333333335,
        "height": 0.7685185185185185
    },
    {
        "class_id": 1,
        "x_center": 0.2296875,
        "y_center": 0.15,
        "width": 0.13125,
        "height": 0.24259259259259258
    },
    {
        "class_id": 2,
        "x_center": 0.71640625,
        "y_center": 0.14675925925925926,
        "width": 0.06197916666666667,
        "height": 0.10833333333333334
    }
]

TXT CONTENT:
3 0.89375 0.37

In [20]:
import xml.etree.ElementTree as ET

with zipfile.ZipFile(annotation_zip, "r") as z:

    # Use the same sample filename
    stem = Path(json_file).stem

    # Read JSON
    json_data = json.loads(
        z.read(json_file).decode("utf-8")
    )

    # Read TXT
    txt_data = z.read(txt_file).decode("utf-8").strip().splitlines()

    txt_boxes = []
    for line in txt_data:
        values = line.split()
        txt_boxes.append([float(v) for v in values])

    # Read XML
    xml_root = ET.fromstring(
        z.read(xml_file).decode("utf-8")
    )

    xml_boxes = []

    for obj in xml_root.findall("object"):
        class_name = obj.find("name").text

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        xml_boxes.append({
            "class_name": class_name,
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax
        })

    print("Number of boxes:")
    print("JSON:", len(json_data))
    print("TXT :", len(txt_boxes))
    print("XML :", len(xml_boxes))

    print("\nJSON:")
    for item in json_data:
        print(item)

    print("\nTXT:")
    for item in txt_boxes:
        print(item)

    print("\nXML:")
    for item in xml_boxes:
        print(item)

Number of boxes:
JSON: 4
TXT : 4
XML : 4

JSON:
{'class_id': 3, 'x_center': 0.89375, 'y_center': 0.3787037037037037, 'width': 0.21145833333333333, 'height': 0.5314814814814814}
{'class_id': 3, 'x_center': 0.1421875, 'y_center': 0.39537037037037037, 'width': 0.27708333333333335, 'height': 0.7685185185185185}
{'class_id': 1, 'x_center': 0.2296875, 'y_center': 0.15, 'width': 0.13125, 'height': 0.24259259259259258}
{'class_id': 2, 'x_center': 0.71640625, 'y_center': 0.14675925925925926, 'width': 0.06197916666666667, 'height': 0.10833333333333334}

TXT:
[3.0, 0.89375, 0.3787037037037037, 0.21145833333333333, 0.5314814814814814]
[3.0, 0.1421875, 0.39537037037037037, 0.27708333333333335, 0.7685185185185185]
[1.0, 0.2296875, 0.15, 0.13125, 0.24259259259259258]
[2.0, 0.71640625, 0.14675925925925926, 0.06197916666666667, 0.10833333333333334]

XML:
{'class_name': 'Little_mallow', 'xmin': 1513.0, 'ymin': 122.0, 'xmax': 1919.0, 'ymax': 696.0}
{'class_name': 'Little_mallow', 'xmin': 7.0, 'ymin': 12.

In [21]:
from collections import defaultdict

class_mapping = defaultdict(set)

with zipfile.ZipFile(annotation_zip, "r") as z:
    json_files = [
        name for name in z.namelist()
        if name.lower().endswith(".json")]

    for json_name in json_files[:100]:
        data = json.loads(z.read(json_name).decode("utf-8"))

        for obj in data:
            class_mapping[obj["class_id"]].add(json_name)

print("Class IDs found in first 100 JSON files:\n")

for class_id in sorted(class_mapping):
    print(f"Class ID {class_id}: {len(class_mapping[class_id])} files")

Class IDs found in first 100 JSON files:

Class ID 0: 21 files
Class ID 1: 79 files
Class ID 2: 74 files
Class ID 3: 14 files
Class ID 5: 20 files
Class ID 6: 2 files
Class ID 8: 3 files
Class ID 9: 1 files
Class ID 10: 1 files
Class ID 11: 10 files
Class ID 12: 8 files


Extract the raw archives

In [22]:
EXTRACTED_DIR = (PROJECT_ROOT / "01_data" / "extracted")

EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

for zip_path in zip_files:

    # Preserve the top-level ZIP name as a folder
    relative_zip = zip_path.relative_to(RAW_DIR)
    output_dir = EXTRACTED_DIR / relative_zip.with_suffix("")

    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting: {relative_zip}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(output_dir)

    print(f"  → {output_dir}")

print("\nExtraction complete.")

Extracting: Crop with Weeds/Canon Camera_Clicks.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/Canon Camera_Clicks
Extracting: Crop with Weeds/iPhone_Clicks.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/iPhone_Clicks
Extracting: Crop with Weeds/intel Real Sense Depth_Annotations.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Annotations
Extracting: Crop with Weeds/intel Real Sense Depth_Clicks.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Crop with Weeds/intel Real Sense Depth_Clicks
Extracting: Individual Weed Species/16 Classes of Weed_Species.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/Individual Weed Species/16 Classes of Weed_Species
Extracting: UAV.zip
  → /content/drive/MyDrive/dissertation_weed_detection/01_data/extracted/UAV

Extraction complete.


In [23]:
from collections import Counter

print("EXTRACTED DATASET STRUCTURE")
print("=" * 80)

for path in sorted(EXTRACTED_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(EXTRACTED_DIR))

print("\n" + "=" * 80)
print("FILE COUNTS BY EXTENSION")

extensions = Counter()

for path in EXTRACTED_DIR.rglob("*"):
    if path.is_file():
        extensions[path.suffix.lower()] += 1

for ext, count in sorted(extensions.items()):
    print(f"{ext or '[no extension]':10} {count}")

Streaming output truncated to the last 5000 lines.
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108yx7q8p3e482558_171.png_Class_5.png
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108yy3if541012527_107.png_Class_5.png
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108z04vmmi0482558_175.png_Class_5.png
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108z0s5mbb0532545_918.png_Class_5.png
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108z1r667xq432514_891.png_Class_5.png
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/5.Obscure_morning _glory(Ipomoea obscura)/1108z2967i3t082536_648.png_Class_5.png
Individual We

In [24]:
print("EXTRACTED FOLDER STRUCTURE")
print("=" * 80)

for path in sorted(EXTRACTED_DIR.rglob("*")):
    if path.is_dir():
        print(path.relative_to(EXTRACTED_DIR))

EXTRACTED FOLDER STRUCTURE
Crop with Weeds
Crop with Weeds/Canon Camera_Clicks
Crop with Weeds/Canon Camera_Clicks/Canon Camera_Clicks
Crop with Weeds/iPhone_Clicks
Crop with Weeds/iPhone_Clicks/iPhone_Clicks
Crop with Weeds/intel Real Sense Depth_Annotations
Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations
Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/Json
Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/PASCAL_VOC
Crop with Weeds/intel Real Sense Depth_Annotations/intel Real Sense Depth_Annotations/YOLO_darknet
Crop with Weeds/intel Real Sense Depth_Clicks
Crop with Weeds/intel Real Sense Depth_Clicks/intel Real Sense Depth_Clicks
Individual Weed Species
Individual Weed Species/16 Classes of Weed_Species
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species
Individual Weed Species/16 Classes of Weed_Species/Individual Weed_Species/0.Kena_(Commplina_be

In [25]:
# Exact locations based on the extracted structure

IMAGE_DIR = (
    EXTRACTED_DIR
    / "Crop with Weeds"
    / "intel Real Sense Depth_Clicks"
    / "intel Real Sense Depth_Clicks")

JSON_DIR = (
    EXTRACTED_DIR
    / "Crop with Weeds"
    / "intel Real Sense Depth_Annotations"
    / "intel Real Sense Depth_Annotations"
    / "Json")

images = {
    p.stem: p
    for p in IMAGE_DIR.glob("*.jpeg")}

annotations = {
    p.stem: p
    for p in JSON_DIR.glob("*.json")}

image_ids = set(images)
annotation_ids = set(annotations)

missing_annotations = image_ids - annotation_ids
missing_images = annotation_ids - image_ids

print("RealSense images       :", len(images))
print("JSON annotations       :", len(annotations))
print("Images without JSON    :", len(missing_annotations))
print("JSON without image     :", len(missing_images))

if missing_annotations:
    print("\nImages missing annotations:")
    for x in sorted(missing_annotations)[:20]:
        print(x)

if missing_images:
    print("\nAnnotations missing images:")
    for x in sorted(missing_images)[:20]:
        print(x)

if not missing_annotations and not missing_images:
    print("\n✓ PERFECT MATCH: every image has exactly one JSON annotation.")

RealSense images       : 6656
JSON annotations       : 6656
Images without JSON    : 0
JSON without image     : 0

✓ PERFECT MATCH: every image has exactly one JSON annotation.


In [26]:
import pandas as pd

manifest = pd.DataFrame({
    "image_id": sorted(images.keys())})

manifest["image_path"] = manifest["image_id"].map(
    lambda x: str(images[x]))

manifest["annotation_path"] = manifest["image_id"].map(
    lambda x: str(annotations[x]))

manifest["image_filename"] = manifest["image_id"] + ".jpeg"

manifest_path = (
    PROJECT_ROOT
    / "01_data"
    / "processed"
    / "dataset_manifest.csv")

manifest.to_csv(manifest_path, index=False)

print(f"Manifest created: {manifest_path}")
print(f"Number of records: {len(manifest)}")

display(manifest.head())

Manifest created: /content/drive/MyDrive/dissertation_weed_detection/01_data/processed/dataset_manifest.csv
Number of records: 6656


,image_id,image_path,annotation_path,image_filename
0,090800iy3emo472414_684,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090800iy3emo472414_684.jpeg
1,090805j6869f452411_137,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090805j6869f452411_137.jpeg
2,090807o0g127452411_140,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,090807o0g127452411_140.jpeg
3,09080gy3876a462413_765,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,09080gy3876a462413_765.jpeg
4,09080rlexxy5462413_146,/content/drive/MyDrive/dissertation_weed_detec...,/content/drive/MyDrive/dissertation_weed_detec...,09080rlexxy5462413_146.jpeg


In [27]:
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm

widths = []
heights = []
modes = []
failed_images = []

for image_id, image_path in tqdm(
    images.items(),
    total=len(images),
    desc="Checking images"
):
    try:
        with Image.open(image_path) as img:
            widths.append(img.width)
            heights.append(img.height)
            modes.append(img.mode)

    except Exception as e:
        failed_images.append({
            "image_id": image_id,
            "error": str(e)
        })

print("\nIMAGE AUDIT")
print("=" * 60)

print("Total images       :", len(images))
print("Readable images    :", len(widths))
print("Failed images      :", len(failed_images))

print("\nImage modes:")
for mode, count in Counter(modes).items():
    print(f"  {mode}: {count}")

print("\nImage dimensions:")
dimension_counts = Counter(zip(widths, heights))

for (width, height), count in dimension_counts.most_common():
    print(f"  {width} × {height}: {count}")

Checking images:   0%|          | 0/6656 [00:00<?, ?it/s]


IMAGE AUDIT
Total images       : 6656
Readable images    : 6656
Failed images      : 0

Image modes:
  RGB: 6656

Image dimensions:
  1920 × 1080: 6656


In [29]:
class_counts = Counter()
boxes_per_image = []
invalid_annotations = []

for image_id in tqdm(
    sorted(annotations.keys()),
    desc="Analyzing annotations"
):
    annotation_path = annotations[image_id]

    try:
        with open(annotation_path, "r") as f:
            data = json.load(f)

        if not isinstance(data, list):
            invalid_annotations.append(
                (image_id, "Annotation is not a list")
            )
            continue

        boxes_per_image.append(len(data))

        for obj in data:
            required = {
                "class_id",
                "x_center",
                "y_center",
                "width",
                "height"}

            if not required.issubset(obj.keys()):
                invalid_annotations.append(
                    (image_id, "Missing required field"))
                continue

            class_counts[obj["class_id"]] += 1

    except Exception as e:
        invalid_annotations.append(
            (image_id, str(e)))

print("\nANNOTATION AUDIT")
print("=" * 60)

print("Images analyzed       :", len(boxes_per_image))
print("Total bounding boxes  :", sum(boxes_per_image))
print("Invalid annotations    :", len(invalid_annotations))

print("\nClass distribution:")
for class_id, count in sorted(class_counts.items()):
    print(f"  Class {class_id}: {count}")

print("\nBoxes per image:")
print("  Minimum :", min(boxes_per_image))
print("  Maximum :", max(boxes_per_image))
print("  Average :", sum(boxes_per_image) / len(boxes_per_image))

Analyzing annotations:   0%|          | 0/6656 [00:00<?, ?it/s]


ANNOTATION AUDIT
Images analyzed       : 6656
Total bounding boxes  : 62052
Invalid annotations    : 0

Class distribution:
  Class 0: 6512
  Class 1: 22739
  Class 2: 22394
  Class 3: 2217
  Class 4: 1804
  Class 5: 2278
  Class 6: 181
  Class 7: 214
  Class 8: 836
  Class 9: 1425
  Class 10: 476
  Class 11: 126
  Class 12: 399
  Class 13: 184
  Class 14: 267

Boxes per image:
  Minimum : 0
  Maximum : 35
  Average : 9.322716346153847


In [30]:
from collections import Counter

box_count_distribution = Counter(boxes_per_image)

print("NUMBER OF IMAGES BY NUMBER OF BOXES")
print("=" * 60)

for num_boxes, count in sorted(box_count_distribution.items()):
    print(f"{num_boxes:2} boxes : {count:4} images")

print("\nSummary")
print("-" * 60)
print("Images with 0 boxes :", box_count_distribution.get(0, 0))
print("Images with ≥1 box  :", sum(
    count for boxes, count in box_count_distribution.items()
    if boxes >= 1))

NUMBER OF IMAGES BY NUMBER OF BOXES
 0 boxes :    8 images
 1 boxes :  110 images
 2 boxes :  244 images
 3 boxes :  332 images
 4 boxes :  429 images
 5 boxes :  515 images
 6 boxes :  570 images
 7 boxes :  557 images
 8 boxes :  545 images
 9 boxes :  509 images
10 boxes :  463 images
11 boxes :  386 images
12 boxes :  349 images
13 boxes :  303 images
14 boxes :  287 images
15 boxes :  232 images
16 boxes :  193 images
17 boxes :  146 images
18 boxes :  112 images
19 boxes :  102 images
20 boxes :   82 images
21 boxes :   58 images
22 boxes :   44 images
23 boxes :   16 images
24 boxes :   16 images
25 boxes :   15 images
26 boxes :    8 images
27 boxes :    9 images
28 boxes :    5 images
29 boxes :    2 images
30 boxes :    4 images
31 boxes :    3 images
32 boxes :    1 images
35 boxes :    1 images

Summary
------------------------------------------------------------
Images with 0 boxes : 8
Images with ≥1 box  : 6648


In [31]:
XML_DIR = (
    EXTRACTED_DIR
    / "Crop with Weeds"
    / "intel Real Sense Depth_Annotations"
    / "intel Real Sense Depth_Annotations"
    / "PASCAL_VOC")

xml_class_counts = Counter()
xml_files_checked = 0
xml_errors = []

for xml_path in tqdm(
    XML_DIR.glob("*.xml"),
    total=len(list(XML_DIR.glob("*.xml"))),
    desc="Reading XML annotations"
):
    try:
        root = ET.parse(xml_path).getroot()

        for obj in root.findall("object"):
            name = obj.find("name").text
            xml_class_counts[name] += 1

        xml_files_checked += 1

    except Exception as e:
        xml_errors.append((xml_path.name, str(e)))

print("\nXML CLASS AUDIT")
print("=" * 60)

print("XML files checked :", xml_files_checked)
print("XML errors        :", len(xml_errors))
print("Unique class names:", len(xml_class_counts))

print("\nClasses:")
for class_name, count in xml_class_counts.most_common():
    print(f"{class_name:45} {count}")

Reading XML annotations:   0%|          | 0/6656 [00:00<?, ?it/s]


XML CLASS AUDIT
XML files checked : 6656
XML errors        : 0
Unique class names: 15

Classes:
lavhala_cyperus_rotundus                      22891
lambs_quarter_plant                           22523
kena_commplina_benghalensio                   6553
Obscure_morning _glory                        2294
Little_mallow                                 2244
moti_dudhi_euphorbia_geneculata_L             1818
Digitaria_sp                                  1439
choti_dudhi_Euphorbia_hirta                   836
gajar_gavat_congressgavat                     479
Sicklepod                                     403
dwarf_cassia                                  269
bilayat_mexicana_argemone                     214
Asian_pigeonwings                             189
harali_cynodon_dactylon                       184
Graceful_sandmat                              129


In [32]:
# Locate the 16 weed-species folders
SPECIES_ROOT = (
    EXTRACTED_DIR
    / "Individual Weed Species"
    / "16 Classes of Weed_Species"
    / "Individual Weed_Species")

species_folders = sorted([
    p for p in SPECIES_ROOT.iterdir()
    if p.is_dir()])

print("NUMBER OF SPECIES FOLDERS:", len(species_folders))
print("\nSpecies folders:")
print("=" * 80)

for i, folder in enumerate(species_folders):
    print(f"{i:2}: {folder.name}")

print("\n" + "=" * 80)
print("Detection annotation classes:")
print("=" * 80)

for class_name in sorted(xml_class_counts):
    print(class_name)

NUMBER OF SPECIES FOLDERS: 16

Species folders:
 0: 0.Kena_(Commplina_benghalensio)
 1: 1..Lavhala_(Cyperus_Rotundus)
 2: 10.Gajar_gavat_(Parthenium hysterophorus)
 3: 11.Graceful_Sandmart_(Euphorbia hypericifolia)
 4: 12.Sicklepod_(Senna obtusifolia)
 5: 13.Harali_(Cynodon_dactylon)
 6: 14.Dwarf_cassia_(Chamaecrista pumila)
 7: 15.Punarnava _(Boerhaavia diffusa)
 8: 2.Lamber_Quarter_plant(Chenopodium )
 9: 3.Little_Mallow(Malva parviflora)
10: 4.Moti_dudhi(Euphorbia_geneculata_L)
11: 5.Obscure_morning _glory(Ipomoea obscura)
12: 6.Asian_Pigeonwings_(Clitoria Ternatea)
13: 7.Bilayat_(Mexicana_Argemone)
14: 8.Choti_dudhi_(Euphorbia_hirta)
15: 9.Digitaria_SP_(Digitaria Sanguinalis )

Detection annotation classes:
Asian_pigeonwings
Digitaria_sp
Graceful_sandmat
Little_mallow
Obscure_morning _glory
Sicklepod
bilayat_mexicana_argemone
choti_dudhi_Euphorbia_hirta
dwarf_cassia
gajar_gavat_congressgavat
harali_cynodon_dactylon
kena_commplina_benghalensio
lambs_quarter_plant
lavhala_cyperus_rot

In [34]:
# Build class_id -> class_name mapping directly from
# JSON + XML annotations

from collections import defaultdict, Counter

class_id_to_names = defaultdict(set)
class_id_counts = Counter()
pairing_errors = []

xml_files = list(XML_DIR.glob("*.xml"))

for xml_path in tqdm(xml_files, desc="Verifying class mapping"):

    image_id = xml_path.stem
    json_path = annotations.get(image_id)

    if json_path is None:
        pairing_errors.append(
            (image_id, "Missing JSON")
        )
        continue

    # Read JSON
    with open(json_path, "r") as f:
        json_data = json.load(f)

    # Read XML
    root = ET.parse(xml_path).getroot()
    xml_objects = root.findall("object")

    # Number of objects must agree
    if len(json_data) != len(xml_objects):
        pairing_errors.append(
            (
                image_id,
                f"JSON boxes={len(json_data)}, XML boxes={len(xml_objects)}"
            )
        )
        continue

    # Pair objects by position
    for json_obj, xml_obj in zip(json_data, xml_objects):

        class_id = int(json_obj["class_id"])
        class_name = xml_obj.find("name").text.strip()

        class_id_to_names[class_id].add(class_name)
        class_id_counts[class_id] += 1


print("\nCLASS MAPPING VERIFICATION")
print("=" * 80)

print("XML files checked :", len(xml_files))
print("Pairing errors    :", len(pairing_errors))

print("\nClass ID → class name:")
print("-" * 80)

for class_id in sorted(class_id_to_names):

    names = class_id_to_names[class_id]

    print(
        f"Class ID {class_id:2} | "
        f"Instances: {class_id_counts[class_id]:5} | "
        f"Names: {sorted(names)}")

print("\nConsistency check:")
print("-" * 80)

inconsistent = {
    class_id: names
    for class_id, names in class_id_to_names.items()
    if len(names) != 1}

print(
    "Class IDs with multiple names:",
    len(inconsistent))

if inconsistent:
    for class_id, names in inconsistent.items():
        print(f"  Class {class_id}: {sorted(names)}")
else:
    print("✓ Every class ID maps to exactly one class name.")

Verifying class mapping:   0%|          | 0/6656 [00:00<?, ?it/s]


CLASS MAPPING VERIFICATION
XML files checked : 6656
Pairing errors    : 360

Class ID → class name:
--------------------------------------------------------------------------------
Class ID  0 | Instances:  6082 | Names: ['kena_commplina_benghalensio']
Class ID  1 | Instances: 21550 | Names: ['lavhala_cyperus_rotundus']
Class ID  2 | Instances: 20921 | Names: ['Obscure_morning _glory', 'lambs_quarter_plant']
Class ID  3 | Instances:  2063 | Names: ['Little_mallow']
Class ID  4 | Instances:  1631 | Names: ['moti_dudhi_euphorbia_geneculata_L']
Class ID  5 | Instances:  2174 | Names: ['Obscure_morning _glory']
Class ID  6 | Instances:   157 | Names: ['Asian_pigeonwings']
Class ID  7 | Instances:   200 | Names: ['bilayat_mexicana_argemone']
Class ID  8 | Instances:   781 | Names: ['choti_dudhi_Euphorbia_hirta']
Class ID  9 | Instances:  1319 | Names: ['Digitaria_sp']
Class ID 10 | Instances:   456 | Names: ['gajar_gavat_congressgavat']
Class ID 11 | Instances:   117 | Names: ['Graceful_sa

In [35]:
# Find the first annotation file where JSON and XML
# contain a different number of objects.

problem_file = None

for xml_path in XML_DIR.glob("*.xml"):

    image_id = xml_path.stem
    json_path = annotations.get(image_id)

    if json_path is None:
        continue

    with open(json_path, "r") as f:
        json_data = json.load(f)

    root = ET.parse(xml_path).getroot()
    xml_objects = root.findall("object")

    if len(json_data) != len(xml_objects):
        problem_file = image_id
        break

print("First problematic image ID:", problem_file)

if problem_file:
    json_path = annotations[problem_file]
    xml_path = XML_DIR / f"{problem_file}.xml"

    with open(json_path, "r") as f:
        json_data = json.load(f)

    root = ET.parse(xml_path).getroot()
    xml_objects = root.findall("object")

    print("\nJSON objects:", len(json_data))
    print("XML objects :", len(xml_objects))

    print("\nJSON:")
    for i, obj in enumerate(json_data):
        print(i, obj)

    print("\nXML:")
    for i, obj in enumerate(xml_objects):
        name = obj.find("name").text.strip()
        bbox = obj.find("bndbox")

        print(
            i,
            name,
            {
                "xmin": bbox.find("xmin").text,
                "ymin": bbox.find("ymin").text,
                "xmax": bbox.find("xmax").text,
                "ymax": bbox.find("ymax").text,})

First problematic image ID: 090807o0g127452411_140

JSON objects: 8
XML objects : 9

JSON:
0 {'class_id': 1, 'x_center': 0.30755208333333334, 'y_center': 0.11388888888888889, 'width': 0.6151041666666667, 'height': 0.2111111111111111}
1 {'class_id': 1, 'x_center': 0.3533854166666667, 'y_center': 0.9425925925925925, 'width': 0.2421875, 'height': 0.08333333333333333}
2 {'class_id': 1, 'x_center': 0.17734375, 'y_center': 0.41898148148148145, 'width': 0.3546875, 'height': 0.39537037037037037}
3 {'class_id': 1, 'x_center': 0.23359375, 'y_center': 0.7569444444444444, 'width': 0.24010416666666667, 'height': 0.275}
4 {'class_id': 1, 'x_center': 0.44244791666666666, 'y_center': 0.6847222222222222, 'width': 0.10572916666666667, 'height': 0.16203703703703703}
5 {'class_id': 1, 'x_center': 0.64453125, 'y_center': 0.9421296296296297, 'width': 0.0796875, 'height': 0.09351851851851851}
6 {'class_id': 2, 'x_center': 0.6825520833333333, 'y_center': 0.6027777777777777, 'width': 0.0546875, 'height': 0.05}

In [36]:
YOLO_DIR = (
    EXTRACTED_DIR
    / "Crop with Weeds"
    / "intel Real Sense Depth_Annotations"
    / "intel Real Sense Depth_Annotations"
    / "YOLO_darknet")

json_txt_mismatches = []

for image_id in tqdm(
    sorted(images.keys()),
    desc="Comparing JSON and TXT"
):
    json_path = annotations[image_id]
    txt_path = YOLO_DIR / f"{image_id}.txt"

    # JSON
    with open(json_path, "r") as f:
        json_data = json.load(f)

    json_boxes = [
        [
            int(obj["class_id"]),
            float(obj["x_center"]),
            float(obj["y_center"]),
            float(obj["width"]),
            float(obj["height"])
        ]
        for obj in json_data]

    # TXT
    with open(txt_path, "r") as f:
        txt_lines = [
            line.strip()
            for line in f
            if line.strip()]

    txt_boxes = [
        [int(parts[0])] + [float(x) for x in parts[1:]]
        for parts in (line.split() for line in txt_lines)]

    # Compare
    if len(json_boxes) != len(txt_boxes):
        json_txt_mismatches.append(
            (image_id, "different number of boxes")
        )
        continue

    for j_box, t_box in zip(json_boxes, txt_boxes):
        if j_box != t_box:
            json_txt_mismatches.append(
                (image_id, "box values differ")
            )
            break

print("\nJSON ↔ TXT VERIFICATION")
print("=" * 60)

print("Images checked :", len(images))
print("Mismatches     :", len(json_txt_mismatches))

if json_txt_mismatches:
    print("\nFirst 20 mismatches:")
    for item in json_txt_mismatches[:20]:
        print(item)
else:
    print("\n✓ JSON and TXT annotations are identical for every image.")

Comparing JSON and TXT:   0%|          | 0/6656 [00:00<?, ?it/s]


JSON ↔ TXT VERIFICATION
Images checked : 6656
Mismatches     : 0

✓ JSON and TXT annotations are identical for every image.


In [39]:
class_image_counts = Counter()
class_box_counts = Counter()
class_image_ids = defaultdict(set)

for image_id in tqdm(
    sorted(images.keys()),
    desc="Analyzing class distribution"
):
    with open(annotations[image_id], "r") as f:
        data = json.load(f)

    classes_in_image = set()

    for obj in data:
        class_id = int(obj["class_id"])

        class_box_counts[class_id] += 1
        classes_in_image.add(class_id)

    for class_id in classes_in_image:
        class_image_counts[class_id] += 1
        class_image_ids[class_id].add(image_id)

print("\nCLASS DISTRIBUTION")
print("=" * 75)

print(
    f"{'Class ID':<10}"
    f"{'Images':>12}"
    f"{'Boxes':>12}"
    f"{'Boxes/Image':>15}"
)

print("-" * 75)

for class_id in sorted(class_box_counts):

    images_with_class = class_image_counts[class_id]
    boxes = class_box_counts[class_id]

    print(
        f"{class_id:<10}"
        f"{images_with_class:>12}"
        f"{boxes:>12}"
        f"{boxes / images_with_class:>15.2f}")

Analyzing class distribution:   0%|          | 0/6656 [00:00<?, ?it/s]


CLASS DISTRIBUTION
Class ID        Images       Boxes    Boxes/Image
---------------------------------------------------------------------------
0                 3377        6512           1.93
1                 5059       22739           4.49
2                 5546       22394           4.04
3                 1356        2217           1.63
4                 1021        1804           1.77
5                 1091        2278           2.09
6                  173         181           1.05
7                  186         214           1.15
8                  615         836           1.36
9                  721        1425           1.98
10                 343         476           1.39
11                  88         126           1.43
12                 333         399           1.20
13                  55         184           3.35
14                 228         267           1.17


In [40]:
from itertools import combinations
cooccurrence = Counter()

for image_id in tqdm(
    sorted(images.keys()),
    desc="Analyzing class co-occurrence"
):
    with open(annotations[image_id], "r") as f:
        data = json.load(f)

    classes = sorted({
        int(obj["class_id"])
        for obj in data
    })

    for pair in combinations(classes, 2):
        cooccurrence[pair] += 1

print("\nCLASS CO-OCCURRENCE")
print("=" * 60)

print("Most common class pairs:")
print("-" * 60)

for (class_a, class_b), count in cooccurrence.most_common(20):
    print(
        f"Class {class_a:2} + Class {class_b:2} : "
        f"{count:4} images")

Analyzing class co-occurrence:   0%|          | 0/6656 [00:00<?, ?it/s]


CLASS CO-OCCURRENCE
Most common class pairs:
------------------------------------------------------------
Class  1 + Class  2 : 4136 images
Class  0 + Class  2 : 2923 images
Class  0 + Class  1 : 2652 images
Class  2 + Class  3 : 1141 images
Class  1 + Class  3 :  992 images
Class  2 + Class  5 :  918 images
Class  2 + Class  4 :  857 images
Class  0 + Class  3 :  786 images
Class  1 + Class  5 :  782 images
Class  1 + Class  4 :  739 images
Class  0 + Class  5 :  650 images
Class  2 + Class  9 :  601 images
Class  2 + Class  8 :  547 images
Class  1 + Class  8 :  445 images
Class  0 + Class  4 :  443 images
Class  1 + Class  9 :  373 images
Class  0 + Class  8 :  356 images
Class  2 + Class 10 :  315 images
Class  3 + Class  4 :  304 images
Class  2 + Class 12 :  271 images


In [41]:
image_ids = sorted(images.keys())

# Show representative filenames
print("Sample image IDs:")
print("=" * 60)

for image_id in image_ids[:30]:
    print(image_id)

print("\n" + "=" * 60)
print("Filename lengths:")

lengths = Counter(len(x) for x in image_ids)

for length, count in sorted(lengths.items()):
    print(f"{length} characters : {count} images")

Sample image IDs:
090800iy3emo472414_684
090805j6869f452411_137
090807o0g127452411_140
09080gy3876a462413_765
09080rlexxy5462413_146
09080x5380ao462413_166
0908128u2u45452411_747
090815j0fsy5452411_155
090819o95vx6452411_109
09082097wg80452411_110
09082230r1jf472414_145
0908231qlh77452411_161
09082i79k833452411_720
09082l7en507472414_414
09082nc82173452411_774
09082wkrvdnc472414_468
09082wvvud0m452411_144
090831w0h314462413_774
090833n38m4y472414_666
090836i0a530452411_666
09083i768rwt452411_163
09083k586x3z472414_522
09083s11evtn452411_145
09083trb6yj5472414_63.
09083urg186k452411_162
0908421t941n462413_144
0908458cmi4a472414_801
090848y2w08g472414_477
09084s1c106d462413_171
09084y23cc1a462413_145

Filename lengths:
22 characters : 6656 images


In [42]:
ids = sorted(images.keys())

print("POSITION-WISE FILENAME ANALYSIS")
print("=" * 70)

for pos in range(len(ids[0])):

    values = [image_id[pos] for image_id in ids]
    counts = Counter(values)

    print(
        f"Position {pos:2}: "
        f"{len(counts):3} unique values | "
        f"Most common: {counts.most_common(5)}")

POSITION-WISE FILENAME ANALYSIS
Position  0:   2 unique values | Most common: [('1', 6521), ('0', 135)]
Position  1:   8 unique values | Most common: [('1', 3529), ('2', 1530), ('0', 632), ('5', 423), ('3', 249)]
Position  2:   1 unique values | Most common: [('0', 6656)]
Position  3:   1 unique values | Most common: [('8', 6656)]
Position  4:  36 unique values | Most common: [('4', 319), ('3', 308), ('9', 304), ('8', 296), ('2', 290)]
Position  5:  36 unique values | Most common: [('9', 311), ('0', 309), ('1', 308), ('5', 305), ('8', 302)]
Position  6:  36 unique values | Most common: [('6', 299), ('4', 296), ('3', 293), ('1', 291), ('7', 291)]
Position  7:  36 unique values | Most common: [('3', 316), ('9', 308), ('2', 308), ('7', 306), ('0', 297)]
Position  8:  36 unique values | Most common: [('4', 309), ('9', 305), ('3', 298), ('6', 296), ('8', 295)]
Position  9:  36 unique values | Most common: [('1', 329), ('4', 303), ('7', 295), ('0', 292), ('2', 282)]
Position 10:  36 unique v

In [43]:
import hashlib
from collections import defaultdict

image_hashes = defaultdict(list)

for image_id in tqdm(
    sorted(images.keys()),
    desc="Hashing images"
):
    image_path = images[image_id]

    with open(image_path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    image_hashes[file_hash].append(image_id)

duplicate_groups = {
    h: ids
    for h, ids in image_hashes.items()
    if len(ids) > 1
}

print("\nEXACT DUPLICATE ANALYSIS")
print("=" * 60)

print("Total images          :", len(images))
print("Unique file hashes    :", len(image_hashes))
print("Duplicate groups      :", len(duplicate_groups))

duplicate_images = sum(
    len(ids) for ids in duplicate_groups.values()
)

print("Images involved       :", duplicate_images)

if duplicate_groups:
    print("\nFirst 10 duplicate groups:")
    for i, ids in enumerate(duplicate_groups.values()):
        print(f"{i+1}: {ids}")
        if i == 9:
            break
else:
    print("\n✓ No exact duplicate images found.")

Hashing images:   0%|          | 0/6656 [00:00<?, ?it/s]


EXACT DUPLICATE ANALYSIS
Total images          : 6656
Unique file hashes    : 6656
Duplicate groups      : 0
Images involved       : 0

✓ No exact duplicate images found.
